In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from matplotlib import colors as mcolors

import package_DBR.py
from package_DBR import myRound, SelectPath_RT, Delay_RT, FO_RT, FOPDT, SOPDT, FOPDT_cost, SOPDT_cost, Process, Bode


ModuleNotFoundError: No module named 'package_DBR'

In [ ]:
nameFile = 'Cleaned_data_Open_loop_experiment_on_MV_2026-03-03-09h18.txt'

titleName = nameFile.split('.')[0]    
data = pd.read_csv('Data/' + nameFile)

if 'MV' in nameFile:
    ExpVariable = 'MV'
    tm = data['tm'].values
    MVm = data['MVm'].values
    PVm = data['PVm'].values    
else:    
    ExpVariable = 'DV'
    tm = data['tm'].values
    DVm = data['DVm'].values 
    PVm = data['PVm'].values
     
print(ExpVariable)    

MV


In [ ]:
# --- 1. Paramètres de Simulation (Test Boucle Ouverte du PID) ---
TSim = 250.0
Ts   = 0.1
N    = int(TSim / Ts) + 1

# Réglages PID (estimés d'après la pente de ton MVI et le pic du MVD)
Kc, Ti, Td = IMC_tuning(0.32, 149.5, 15.0, 4.0, 0.5)
alpha = 0.5
MVMin, MVMax = 0.0, 100.0

# --- 2. Initialisation des Vecteurs ---
t, SP, PV, E = [], [], [], []
MV, MVP, MVI, MVD, MVFF = [], [], [], [], []
Man, MVMan, ManFF = [], [], []

# --- 3. Boucle de Simulation ---
for i in range(N):
    curr_t = i * Ts
    t.append(curr_t)
    
    # PV reste constant à 50 (boucle ouverte, pas de modèle de procédé)
    pv_val = 50.0
    PV.append(pv_val)
    
    # Setpoint (SP) : Échelon de 50 à 60 à t=10s
    sp_val = 60.0 if curr_t >= 10 else 50.0
    SP.append(sp_val)
    
    # Mode Manuel (Man) : Actif de 100s à 120s, puis à nouveau après 230s
    is_man = True if (100 <= curr_t < 120) or (curr_t >= 230) else False
    Man.append(is_man)
    
    # Valeur forcé en Manuel (MVMan) : ~42% au premier créneau, ~35% au second
    mv_man_val = 42.0 if curr_t < 230 else 35.0
    MVMan.append(mv_man_val)
    
    # Feedforward (MVFF) : Échelon à 10% à partir de t=200s
    mv_ff_val = 10.0 if curr_t >= 200 else 0.0
    MVFF.append(mv_ff_val)
    
    # Mode Feedforward Manuel (ManFF) : Actif à partir de 240s
    is_man_ff = True if curr_t >= 240 else False
    ManFF.append(is_man_ff)

    # 4. APPEL DE TA FONCTION PID EN TEMPS RÉEL
    PID_RT(SP, [pv_val], Man, MVMan, [mv_ff_val], 
           Kc, Ti, Td, alpha, Ts, 
           MVMin, MVMax, 
           MV, MVP, MVI, MVD, E, 
           ManFF=is_man_ff, PVInit=50.0)

# --- 4. Affichage (Reproduction fidèle de ton image) ---
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Subplot 1 : Sorties du contrôleur (MV, MVP, MVI, MVD, MVFF)
ax1.plot(t, MV,   'b-',  label='MV', linewidth=2)
ax1.plot(t, MVP,  'gray',label='MVP', alpha=0.8)
ax1.plot(t, MVI,  'g-',  label='MVI', linewidth=1.5)
ax1.plot(t, MVD,  'c-',  label='MVD', alpha=0.8)
ax1.plot(t, MVFF, 'r-',  label='MVFF', alpha=0.6)
ax1.set_ylabel('Value of MV, MVP, MVI, MVD & MVFF [%]')
ax1.set_ylim([-10, 105])
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_title('Step response of the PID controller')

# Subplot 2 : Variables du procédé et Erreur (SP, PV, E)
ax2.step(t, SP, 'r-', label='SP', where='post', linewidth=1.5)
ax2.plot(t, PV, 'g-', label='PV', linewidth=1.5)
ax2.step(t, E,  'k-', label='E',  where='post', linewidth=1.5)
ax2.set_ylabel('Value of SP, PV & E [°C]')
ax2.set_ylim([-5, 105])
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

# Subplot 3 : États booléens (Man et ManFF) allant de 0 à 1
ax3.step(t, Man,   'r-', label='Man',   where='post', linewidth=1.5)
ax3.step(t, ManFF, 'orange', label='ManFF', where='post', linewidth=1.5)
ax3.set_ylabel('Value of Man/FF')
ax3.set_xlabel('Time [s]')
ax3.set_ylim([-0.2, 1.2])
ax3.set_yticks([0, 1])
ax3.legend(loc='center left')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

SyntaxError: invalid syntax (1014939936.py, line 17)